#CS-215 Final Project - Main Colab

This Google Colab notebook presents a data science analysis of ChatGPT usage patterns across different academic majors, including Math/Engineering, Physics, and Biology. The project explores how students interact with ChatGPT using real conversation data, focusing on differences in usage frequency, interaction style, temporal behavior, and prompt complexity.

The analysis includes data preprocessing, feature engineering, clustering of user prompts using KMeans, and multiple visualizations such as usage comparisons, time-based trends, and cluster distributions.

The goal of this project is to identify and compare behavioral patterns in how students from different disciplines use ChatGPT, and to understand how academic background influences AI-assisted learning behavior.

##Research Questions

1. How does ChatGPT usage volume vary across academic majors?

2. How do interaction patterns with ChatGPT differ across majors?

3. How do temporal usage patterns of ChatGPT differ across majors?

4. How does prompt complexity vary across majors?



##Data Preparation & Data-Preprocessing

This code combines multiple datasets from different academic majors into one unified dataset, then filters it by a specific time range for analysis.






In [ ]:
#Import necessary libraries
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.cluster import KMeans
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

First, it loads six separate CSV files, each representing ChatGPT user data from different majors (Math-Engineering, Physics, Biology, Physics-Math, Physics-Engineering, and Computer Science). It then adds a new column called "major" to each dataset so that each row is labeled with its corresponding academic field.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
math_engineering_df = pd.read_csv("/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/math_engineering_user_df.csv")
physics_df = pd.read_csv("/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/physics_user_df.csv")
biology_df = pd.read_csv("/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/bio_user_df.csv")
physics_math_df = pd.read_csv("/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/physics_math_user_df.csv")
physics_engineering_df = pd.read_csv("/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/physics_engineering_user_df.csv")
cs_df = pd.read_csv("/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/cs_user_df.csv")

In [ ]:
math_engineering_df["major"] = "Math-engineering"
physics_df["major"] = "Physics"
biology_df["major"] = "Biology"
physics_math_df["major"] = "Physics-math"
physics_engineering_df["major"] = "Physics-engineering"
cs_df["major"] = "Computer Science"

Next, all datasets are merged into a single dataframe (all_df) using concatenation. This creates one combined dataset containing users from all majors.

In [ ]:
all_df = pd.concat(
    [math_engineering_df, physics_df, biology_df, physics_math_df, physics_engineering_df, cs_df],
    ignore_index=True
)

After merging, the code converts the timestamp column into a proper datetime format so it can be used for time-based filtering.

Then, it filters the data to keep only records between September 13, 2025 and March 31, 2026, creating a cleaned subset called filtered_df.

In [ ]:
all_df["timestamp"] = pd.to_datetime(all_df["timestamp"])

start_date = "2025-9-13"
end_date = "2026-03-31"

filtered_df = all_df[
    (all_df["timestamp"] >= start_date) &
    (all_df["timestamp"] <= end_date)
].copy()

In [ ]:
all_df.head()

,conversation_id,role,text,timestamp,date,hour,day_of_week,month,length,major
0,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,Week 7: Frictional and Normal Forces Problem (...,2025-03-05 14:32:43.802000046,2025-03-05,14,Wednesday,2025-03,1758,Math-engineering
1,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,write up,2025-03-05 14:33:06.992000103,2025-03-05,14,Wednesday,2025-03,8,Math-engineering
2,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,\tTrial 1\tTrial 2\tTrial 3\tTrial 4\tAverage\...,2025-03-05 14:59:52.551000118,2025-03-05,14,Wednesday,2025-03,449,Math-engineering
3,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,angle was 0,2025-03-05 15:00:10.618000031,2025-03-05,15,Wednesday,2025-03,11,Math-engineering
4,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,\tTrial 1\tTrial 2\tTrial 3\tTrial 4\tAverage\...,2025-03-05 15:00:30.641000032,2025-03-05,15,Wednesday,2025-03,461,Math-engineering


Finally, it saves this filtered dataset as a CSV file (filtered_df.csv) and prepares it for download in Google Colab. The last line sets df = filtered_df so the filtered dataset is used for further analysis.

In [ ]:
filtered_df.to_csv("filtered_df.csv", index=False)

from google.colab import files
#files.download("filtered_df.csv")

In [ ]:
df = filtered_df

##1. Usage Volume

###How does ChatGPT usage volume vary across academic majors?

This bar chart shows the total number of ChatGPT user messages across different academic majors from Sep 2025 to Mar 2026.

In [ ]:
filtered_df.groupby("major").size()

,0
major,
Biology,286
Computer Science,57
Math-engineering,2121
Physics,831
Physics-engineering,1656
Physics-math,325


In [ ]:
usage = df.groupby("major").size().reset_index(name="count")

fig1 = px.bar(
    usage,
    x="major",
    y="count",
    title="ChatGPT Usage by Major (Sep 2025 – Mar 2026)",
    labels={
        "major": "Major",
        "count": "Number of Messages"
    }
)

fig1.update_layout(
    xaxis_tickangle=30
)

fig1.show()
fig1.write_html("usage_volume.html")

Key takeaway: A Math-engineering student had the highest usage with 2,121 messages, followed by Physics-engineering (1,656) and Physics (831). Biology (286), Physics-math (325), and Computer Science (57) show comparatively lower message counts. Overall, the graph highlights clear differences in ChatGPT usage intensity across majors, with engineering-related tracks showing the highest engagement.

##2. Interaction Patterns

### How do interaction patterns with ChatGPT differ across majors?

The Plotly treemap creates an interactive visualization of ChatGPT usage patterns by major. Each section represents a major and its associated interaction types, with size based on proportion. This makes it easier to visually compare how different majors use ChatGPT.

The TF-IDF vectorizer converts conversation text into numerical data that machine learning models can understand. It identifies the most important words in each conversation while reducing the importance of commonly repeated words. This allows the analysis to focus on meaningful language patterns in ChatGPT interactions.

In [ ]:
vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(filtered_df["text"])

K-Means clustering groups similar conversations together based on their text content. In this analysis, the model creates five clusters representing different types of ChatGPT usage behaviors. This helps identify common interaction patterns among students without manually labeling the data.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
filtered_df["cluster"] = kmeans.fit_predict(X)

Sample conversations from each cluster are printed to help interpret what each group represents. By reviewing these examples, meaningful labels can be assigned to describe the overall theme of each cluster. This step connects the machine learning output to real user behavior.

In [ ]:
for c in sorted(df["cluster"].unique()):
    print("\n------------")
    print(f"CLUSTER {c}")
    print("------------")
    print(df[df["cluster"] == c]["text"].head().values)


------------
CLUSTER 0
------------
['what is the definition of initial value problem in differential equation?'
 'ok how is this happening' 'how is this working like this'
 'for differential equation, if it is linear, is it separable? or if it separable, is it linea'
 'y\n′′′ + 2y\n′′\n− ln(x)y\n′ + y= π why is this linear']

------------
CLUSTER 1
------------
['10. The function s(t) represents the height in meters of a parachutist t seconds into their descent.\nt (s) 0 10 20 30 40 50 60\ns(t) (m) 3000 2100 1600 1350 1100 850 600\na) Find the Average velocity of s(t) on the interval [20, 40].\nb) Use the Central Difference method to approximate s′(20), s′(30), and s′(40).\nc) Use the information from part (a) to estimate s′′(30) using the Central Difference method.\nd) What units does s′′(30) have? Explain the significance of s′′(30) in relation to the velocity,\nthat is, in relation to s′(30).\n7 use the same format but change the numbers'
 '10. The function s(t) represents the hei

The cluster numbers are replaced with descriptive labels such as conceptual learning, problem solving, or study guidance. This makes the results easier to understand and present in the final analysis. The labels summarize the primary purpose of conversations within each cluster.

In [ ]:
cluster_map = {
    0: "Conceptual STEM Understanding",
    1: "Applied Problem Solving & Lab Reports",
    2: "Step-by-Step Learning & Clarification",
    3: "Advanced Equation Solving Practice",
    4: "Strategy, Interpretation & Study Guidance",
}

df["cluster_label"] = df["cluster"].map(cluster_map)

The crosstab calculates the proportion of each ChatGPT usage pattern within every major. Instead of raw counts, it shows percentages, making comparisons between majors more meaningful. This reveals how students from different academic fields tend to use ChatGPT differently.

The dataset is converted into a long format structure so it can be used for visualization in Plotly. This restructuring organizes the data into categories and proportions suitable for interactive charts. It prepares the dataset for clearer visual analysis.

In [ ]:
# Crosstab proportions
cluster_dist = pd.crosstab(
    df["major"],
    df["cluster_label"],
    normalize="index"
)

# Convert to long format
cluster_long = cluster_dist.reset_index().melt(
    id_vars="major",
    var_name="cluster_label",
    value_name="proportion"
)

# Treemap
fig2 = px.treemap(
    cluster_long,
    path=["major", "cluster_label"],   # hierarchy
    values="proportion",
    color="cluster_label",
    title="ChatGPT Usage Patterns by Major (Sep 2025 – Mar 2026)"

)

fig2.update_traces(
    textinfo="label+percent parent"
)

fig2.show()

#fig2.write_html("interaction_patterns.html")



In [ ]:
dominant = df.groupby("major")["cluster_label"].agg(lambda x: x.value_counts().idxmax())
print(dominant)

major
Biology                Applied Problem Solving & Lab Reports
Computer Science       Step-by-Step Learning & Clarification
Math-engineering       Step-by-Step Learning & Clarification
Physics                Step-by-Step Learning & Clarification
Physics-engineering    Step-by-Step Learning & Clarification
Physics-math           Step-by-Step Learning & Clarification
Name: cluster_label, dtype: object


Key takeaway: Most majors rely heavily on Step-by-Step Learning & Clarification and Applied Problem Solving & Lab Reports, indicating that students primarily use ChatGPT for guided help and practical assignments rather than purely conceptual or advanced equation solving tasks. Math-engineering (58.84%) and Physics (56.92%) show strong preference for step-by-step explanations, while Biology (53.85%) leans more toward applied lab/report-style problem solving. More advanced or specialized clusters like Advanced Equation Solving Practice and Strategy/Interpretation remain relatively small across all majors, suggesting they are less common use cases overall.


##3. Temporal Behavior

### How do temporal usage patterns of ChatGPT differ across majors?

This code creates an interactive line graph showing when students from different majors use ChatGPT throughout the day. It calculates the proportion of usage for each hour, reshapes the data for visualization, and plots hourly usage patterns so majors can be compared easily.


In [ ]:
time_dist = pd.crosstab(df["major"], df["hour"], normalize="index")

time_long = time_dist.reset_index().melt(
    id_vars="major",
    var_name="hour",
    value_name="proportion"
)

fig3 = px.line(
    time_long,
    x="hour",
    y="proportion",
    color="major",
    markers=True,
    title="Hourly ChatGPT Usage Pattern by Major",
    labels={
        "hour": "Hour of Day",
        "proportion": "Proportion of Usage",
        "major": "Major"
    }
)

fig3.show()
#fig3.write_html("temporal_behavior.html")


Key takeaway: Usage patterns vary slightly by major but show a consistent trend: most activity occurs during daytime and afternoon hours, especially between roughly 12 PM and 6 PM, where proportions tend to peak across multiple majors. Physics-engineering and Math-engineering students show relatively strong usage in the late afternoon, while Biology and Physics show more spread-out activity with smaller peaks in both morning and afternoon hours. Computer Science has a more uneven pattern due to a smaller sample size, but still follows similar daytime concentration.


##4. Prompt Complexity

### How does prompt complexity vary across majors?

This box plot shows the distribution of prompt lengths across different academic majors, where each point represents how long students' ChatGPT prompts are in terms of text length.

In [ ]:
fig4 = px.box(
    df,
    x="major",
    y="length",
    title="Prompt Length Distribution by Major",
    labels={
        "major": "Major",
        "length": "Prompt Length"
    }
)

fig4.update_layout(
    xaxis_tickangle=30
)

fig4.show()
fig4.write_html("prompt_complexity.html")

In [ ]:
df.groupby("major")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
major,,,,,,,,
Biology,286.0,329.321678,662.172269,1.0,92.00,244.5,327.00,7540.0
Computer Science,57.0,1987.701754,4499.022199,5.0,32.00,50.0,223.00,14680.0
Math-engineering,2121.0,669.328619,2963.002714,1.0,36.00,103.0,346.00,65291.0
Physics,831.0,381.593261,1678.796814,2.0,28.00,54.0,119.00,22671.0
Physics-engineering,1656.0,194.045894,885.852948,1.0,23.75,46.0,94.25,11484.0
Physics-math,325.0,325.575385,904.971600,2.0,21.00,48.0,155.00,6950.0


A Computer Science student tends to write the longest prompts on average, with a very high mean and wide variability, suggesting a mix of short queries and extremely long, detailed inputs. Math-engineering also shows relatively long and highly variable prompts, including some very extreme outliers. In contrast, Physics-engineering and Biology students generally write shorter and more consistent prompts. Physics and Physics-math fall in between, showing moderate prompt lengths with some variability.


##Extra - Heatmap of ChatGPT usage Patterns

This code creates a heatmap that compares how different majors use ChatGPT across various usage categories. It first computes a normalized cross-tabulation so that each major’s values sum to 1, allowing for fair comparison of usage proportions instead of raw counts. The resulting table is then visualized using a heatmap, where darker shades represent higher proportions of usage in a given category.

In [ ]:
heat = pd.crosstab(df["major"], df["cluster_label"], normalize="index")

fig = px.imshow(
    heat,
    text_auto=True,
    color_continuous_scale="Blues",
    aspect="auto"
)

fig.update_layout(
    title="Normalized ChatGPT Usage Patterns by Major",
    xaxis_title="Usage Type",
    yaxis_title="Major"
)

fig.show()

Key takeaway: The results show that the most common usage pattern across most of the majors is “Step-by-Step Learning & Clarification,” indicating that students primarily use ChatGPT for guided explanations and learning support. This category is especially high for Math-engineering and Physics students, suggesting a strong focus on problem-solving assistance and understanding technical material step by step.

Biology stands out by having the highest proportion in “Applied Problem Solving & Lab Reports,” which suggests that students in this major rely more on ChatGPT for structured writing tasks and lab-related academic work. In contrast, Computer Science and Math-engineering rely less on this category compared to Biology.

##Sources, Citations, and Tools Used

This project uses Python-based data analysis and visualization tools to process and analyze ChatGPT conversation datasets. The following external resources, libraries, and tools were used and are credited below:

Libraries & Tools:

- Python (data processing and analysis)
- pandas (data manipulation)
- scikit-learn (text clustering using TF-IDF and KMeans)
- plotly (interactive visualizations)
- seaborn and matplotlib (supporting visualizations)

References & Documentation:

- scikit-learn documentation: https://scikit-learn.org/stable/
- Reddit discussion on ChatGPT JSON archives: https://www.reddit.com/r/OpenAI/comments/1i414z4/any_tips_on_making_json_conversation_archive/
- ConvoViz GitHub project: https://github.com/mohamed-chs/convoviz#readme

Dataset Source:

ChatGPT conversation logs collected and compiled for academic analysis (CS-215 project dataset)

Acknowledgement of External Help:

- Assistance from ChatGPT was used to support debugging.

- Assistance from Professor W-B for use of treemap.

- No external private datasets or non-public sources were used.